#CafChem tools for running a basic chat loop with a small HuggingFace model

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MauricioCafiero/CafChem_Teaching_Notebooks/blob/main/notebooks/HF_agent_step0_CafChem.ipynb)

## This notebook allows you to:
- Load a small (sub-1GB) instruct model from HuggingFace (SmolLM2-360M-Instruct).
- Run a basic chat loop: user message in, model reply out, conversation remembered.

## Requirements:
- Runs quickly on any Colab runtime (GPU or CPU); the model is only ~720MB.

### import libraries

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


### set up model

In [ ]:
# SmolLM2-360M-Instruct: Hugging Face's own small chat model, ~360M
# parameters, ~720MB download (bf16 weights).
# MODEL_REPO = "HuggingFaceTB/SmolLM2-135M-Instruct"  # even smaller (~270MB), noticeably dumber
MODEL_REPO = "HuggingFaceTB/SmolLM2-360M-Instruct"
MODEL_DIR = "models"
MAX_NEW_TOKENS = 512

SYSTEM_PROMPT = "You are a helpful assistant running locally on the user's computer."

print("Loading model (first run downloads ~1GB, then it's cached)...")

device = "cpu"
tokenizer = AutoTokenizer.from_pretrained(MODEL_REPO, cache_dir=MODEL_DIR)
model = AutoModelForCausalLM.from_pretrained(MODEL_REPO, cache_dir=MODEL_DIR, dtype="auto").to(device)

print("Model loaded\n")


### chat loop

In [ ]:
# The conversation so far. The system prompt sets the bot's personality;
# every user and assistant message gets appended here.
messages = [{"role": "system", "content": SYSTEM_PROMPT}]

print("Ready. Type a message (or 'quit').\n")

while True:
	user_input = input("you: ").strip()

	if user_input.lower() in ("quit", "exit"):
		break
	if not user_input:
		continue

	# 1. Add the user's message to the conversation
	messages.append({"role": "user", "content": user_input})

	# 2. Turn the conversation into model input and generate a reply
	inputs = tokenizer.apply_chat_template(
		messages, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(device)
	output_ids = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, pad_token_id=tokenizer.pad_token_id)

	# 3. Decode only the new tokens (everything after the prompt) back to text
	content = tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True).strip()

	# 4. Remember the reply so the model has context for the next turn
	messages.append({"role": "assistant", "content": content})

	print(f"bot: {content}\n")